# 🔬 Ablation Part 2/3 — Mamba-PFAN Scan-Direction Study

**Modes handled by this notebook:** `horizontal_bidirectional`, `vertical_bidirectional`

### ⚙️ Kaggle Setup Checklist
1. **Accelerator** → GPU T4 x1 or P100 *(Notebook options → Accelerator)*
2. **Internet** → ON *(Notebook options → Settings → Internet → ON)*
3. **Dataset** → Add your desmoking dataset via **+ Add Input** (mounts under `/kaggle/input`)


In [ ]:
# ── Step 1: Environment Check & Config ──────────────────────────────────
import os, sys
from pathlib import Path
import torch

# GPU check
if torch.cuda.is_available():
    GPU_IDS = '0'
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
else:
    GPU_IDS = '-1'
    print('⚠️  No GPU — training will be very slow. Enable GPU in Notebook settings.')

# Paths
INPUT_ROOT      = Path('/kaggle/input')
WORK_ROOT       = Path('/kaggle/working/mamba_pfan')
CHECKPOINT_ROOT = WORK_ROOT / 'checkpoints'
REPO_ROOT       = WORK_ROOT / 'mamba_repo'

WORK_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

# Hyperparameters
EPOCHS        = 50
BATCH_SIZE    = 4
PRINT_FREQ    = 20
SKIP_EXISTING = True
MODES         = ["horizontal_bidirectional", "vertical_bidirectional"]

print('Modes assigned to this notebook:', MODES)
print('Available input datasets:', [p.name for p in INPUT_ROOT.iterdir() if p.is_dir()])


In [ ]:
# ── Step 2: Install Dependencies & Clone Repo ───────────────────────────
import subprocess, shutil

%pip install -q timm einops scikit-image Pillow numpy dominate

REPO_URL    = 'https://github.com/GladwinDaniel/Desmoking.git'
REPO_BRANCH = 'main'

if (REPO_ROOT / '.git').is_dir():
    print('Repo already cloned — pulling latest...')
    try:
        subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_ROOT, check=True)
        subprocess.run(['git', 'reset', '--hard', f'origin/{REPO_BRANCH}'], cwd=REPO_ROOT, check=True)
        print('Updated to latest commit.')
    except Exception as e:
        print(f'Pull failed ({e}), re-cloning...')
        shutil.rmtree(REPO_ROOT)
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
else:
    if REPO_ROOT.exists(): shutil.rmtree(REPO_ROOT)
    print(f'Cloning {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
    print('Cloned successfully.')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
%cd {REPO_ROOT}
print('Working dir:', Path.cwd())


In [ ]:
# ── Step 3: Auto-discover Dataset from /kaggle/input ────────────────────
import zipfile

def find_dataset_root(root):
    for cur, dirs, _ in os.walk(root):
        cur = Path(cur)
        if (cur / 'train').is_dir():
            return cur
        if (cur / 'trainA').is_dir() and (cur / 'trainB').is_dir():
            return cur
    return None

DATA_ROOT = find_dataset_root(INPUT_ROOT)

# Fallback: extract zip if dataset not pre-extracted
if DATA_ROOT is None:
    zips = list(INPUT_ROOT.rglob('*.zip'))
    if zips:
        extract_dir = Path('/kaggle/working/extracted_dataset')
        extract_dir.mkdir(parents=True, exist_ok=True)
        print(f'Extracting {zips[0]} ...')
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(extract_dir)
        DATA_ROOT = find_dataset_root(extract_dir)

if DATA_ROOT is None:
    raise FileNotFoundError(
        f'Cannot find dataset under {INPUT_ROOT}.\n'
        'Expected a folder containing train/ or trainA/ + trainB/.'
    )

print(f'✅ Dataset root: {DATA_ROOT}')


In [ ]:
# ── Step 4: Run Training — Part 2/3 (horiz_bidir + vert_bidir) ────────────────

for mode in MODES:
    ckpt = CHECKPOINT_ROOT / f'mamba_scan_{mode}' / 'latest_net_G.pth'
    if SKIP_EXISTING and ckpt.is_file():
        print(f'[SKIP] {mode} — checkpoint already exists.')
        continue

    print(f'\n[RUN] Training mode: {mode}')
    cmd = (
        f'python "{REPO_ROOT}/train.py"'
        f' --dataroot "{DATA_ROOT}"'
        f' --name mamba_scan_{mode}'
        f' --model pix2pix'
        f' --netG mamba_pfan'
        f' --netD basic'
        f' --direction AtoB'
        f' --dataset_mode aligned'
        f' --norm batch'
        f' --checkpoints_dir "{CHECKPOINT_ROOT}"'
        f' --embed_dim 64'
        f' --ndf 64'
        f' --ngf 64'
        f' --batch_size {BATCH_SIZE}'
        f' --num_threads 2'
        f' --n_epochs {EPOCHS}'
        f' --n_epochs_decay 0'
        f' --save_epoch_freq 5'
        f' --display_id -1'
        f' --print_freq {PRINT_FREQ}'
        f' --scan_mode {mode}'
        f' --gpu_ids {GPU_IDS}'
    )
    !{cmd}

print('\n✅ Part 2/3 training complete!')


In [ ]:
# ── Step 5: Evaluate Each Mode (PSNR / SSIM / MSE / CIEDE2000) ───────────
# Finds the test split automatically — looks for test/, testA/testB, or val/
import json

def find_test_dir(data_root):
    data_root = Path(data_root)
    for candidate in ['test', 'val']:
        p = data_root / candidate
        if p.is_dir():
            return p
    # testA/testB side-by-side
    if (data_root / 'testA').is_dir() and (data_root / 'testB').is_dir():
        return data_root
    return data_root   # fall back to root; evaluate_models will raise if no pairs found

TEST_DIR = find_test_dir(DATA_ROOT)
EVAL_ROOT = WORK_ROOT / 'eval_results'
EVAL_SUMMARIES = {}

for mode in MODES:
    ckpt = CHECKPOINT_ROOT / f'mamba_scan_{mode}' / 'latest_net_G.pth'
    if not ckpt.is_file():
        print(f'[SKIP EVAL] No checkpoint for {mode}')
        continue

    eval_out = EVAL_ROOT / mode
    eval_out.mkdir(parents=True, exist_ok=True)
    print(f'\n[EVAL] {mode}  checkpoint={ckpt}  test={TEST_DIR}')

    cmd = (
        f'python "{REPO_ROOT}/evaluate_models.py"'
        f' --model mamba_pfan'
        f' --scan-mode {mode}'
        f' --checkpoint "{ckpt}"'
        f' --test-dir "{TEST_DIR}"'
        f' --output-dir "{eval_out}"'
        f' --device cuda'
    )
    !{cmd}

    summary_path = eval_out / 'evaluation_summary.json'
    if summary_path.is_file():
        EVAL_SUMMARIES[mode] = json.loads(summary_path.read_text())
        m = EVAL_SUMMARIES[mode]['metrics']
        print(f'  PSNR:     {m["psnr_db"]["mean"]:.4f} dB')
        print(f'  SSIM:     {m["ssim"]["mean"]:.4f}')
        print(f'  MSE:      {m["mse"]["mean"]:.4f}')
        print(f'  CIEDE2000:{m["ciede2000"]["mean"]:.4f}')
    else:
        print(f'  WARNING: evaluation_summary.json not found for {mode}')

print('\nAll evaluations complete.')


In [ ]:
# ── Step 6: Save Results & Auto-Download ────────────────────────────────
import json, shutil
from datetime import datetime

def collect_loss_log(checkpoints_dir, name):
    log_path = checkpoints_dir / name / 'loss_log.txt'
    entries = []
    if not log_path.is_file():
        return entries
    for line in log_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('='):
            continue
        try:
            meta_raw, metrics_raw = line.split(')', 1)
            meta = dict(kv.strip().split(': ', 1) for kv in meta_raw.lstrip('(').split(',') if ': ' in kv)
            metrics = dict(kv.strip().split(': ', 1) for kv in metrics_raw.split(',') if ': ' in kv)
            entries.append({**meta, **metrics})
        except Exception:
            pass
    return entries

timestamp = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
output_filename = f'ablation_part2_results_{timestamp}.json'

results = {
    'part': 2,
    'modes_run': MODES,
    'timestamp_utc': timestamp,
    'per_mode_results': {}
}

for mode in MODES:
    run_name = f'mamba_scan_{mode}'
    ckpt_path = CHECKPOINT_ROOT / run_name / 'latest_net_G.pth'
    eval_summary = EVAL_SUMMARIES.get(mode, {})
    results['per_mode_results'][mode] = {
        'checkpoint_exists': ckpt_path.is_file(),
        'checkpoint_path': str(ckpt_path),
        'loss_log': collect_loss_log(CHECKPOINT_ROOT, run_name),
        'eval_metrics': eval_summary.get('metrics', {}),
        'eval_num_images': eval_summary.get('num_images', None),
    }

# Write to /kaggle/working → appears in Kaggle Output panel
out_path = Path(f'/kaggle/working/{output_filename}')
out_path.write_text(json.dumps(results, indent=2))
print(f'Results JSON saved: {out_path}')

# Trigger browser download (Colab)
try:
    from google.colab import files as colab_files
    colab_files.download(str(out_path))
    print('Colab download triggered.')
except (ImportError, Exception):
    pass

print('=' * 60)
print('DONE - Part 2/3 complete!')
print(f'Download your results from the Kaggle Output panel: {output_filename}')
print('=' * 60)
